# Step 4: K-Means Customer Segmentation

We segment customers using **RFM analysis**:
- **Recency** — how many days since their last purchase (lower = more recent = better)
- **Frequency** — how many orders they placed
- **Monetary** — how much total revenue they generated

Then we run K-Means clustering to group similar customers together.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.read_csv('../data/online_retail_clean.csv', parse_dates=['InvoiceDate'])
print('Rows:', df.shape[0])
df.head()

## Build RFM Features

We calculate one row per customer with their Recency, Frequency, and Monetary values.

In [ ]:
# Use the day after the last invoice as our 'today' reference point
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print('Snapshot date:', snapshot_date)

In [ ]:
rfm = df.groupby('Customer ID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('Invoice',     'nunique'),
    Monetary  = ('Revenue',     'sum')
).reset_index()

print('Customers:', rfm.shape[0])
rfm.head()

In [ ]:
rfm.describe()

## Scale the Features

K-Means uses distance between points. If Monetary is in thousands and Recency is in days, the algorithm will be unfairly dominated by the bigger numbers. Scaling fixes this by putting all three features on the same scale.

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])
print('Scaled. Shape:', rfm_scaled.shape)

## Elbow Method — Find the Best Number of Clusters

We try K-Means with 2 to 10 clusters and plot the **inertia** (how tightly packed the clusters are). The "elbow" in the curve tells us the best value of K — the point where adding more clusters stops helping much.

In [ ]:
inertias = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method — Optimal K')
plt.xticks(k_range)
plt.tight_layout()
plt.savefig('../output/elbow_plot.png', dpi=150)
plt.show()
print('Saved elbow plot to output/')

## Run K-Means with K=4

Based on the elbow plot, K=4 is typically a good choice for retail customer data. It gives us four meaningful segments without over-complicating things.

In [ ]:
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = km_final.fit_predict(rfm_scaled)
print('Cluster counts:')
print(rfm['Cluster'].value_counts().sort_index())

## Profile Each Cluster

Now we look at the average Recency, Frequency, and Monetary for each cluster to understand what kind of customer each group represents.

In [ ]:
profile = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean().round(1)
profile['Customer Count'] = rfm.groupby('Cluster')['Customer ID'].count()
profile

## Label the Clusters

Based on the profile above, assign a business-friendly name to each cluster.

Typical patterns:
- **Champions** — bought recently, buy often, spend the most
- **Loyal Customers** — buy often, decent spend, not always recent
- **At-Risk** — haven't bought in a while, used to spend well
- **Low Value** — infrequent, low spend, not recent

Look at the profile table above and update the mapping below to match your actual cluster numbers.

In [ ]:
# Update these numbers based on the profile table output above
cluster_labels = {
    0: 'At-Risk',
    1: 'Champions',
    2: 'Low Value',
    3: 'Loyal Customers'
}

rfm['Segment'] = rfm['Cluster'].map(cluster_labels)
print(rfm['Segment'].value_counts())

## Visualize the Segments

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=rfm, x='Recency', y='Monetary',
    hue='Segment', palette='Set2', alpha=0.6
)
plt.title('Customer Segments — Recency vs Revenue')
plt.tight_layout()
plt.savefig('../output/segments_scatter.png', dpi=150)
plt.show()
print('Saved scatter plot to output/')

## Save the RFM + Segment Data

In [ ]:
rfm.to_csv('../output/rfm_segments.csv', index=False)
print('Saved to output/rfm_segments.csv')
rfm.head()